# Lib-INVENT Scaffold Decoration

Pipeline: **PrexSyn samples → scaffold fragmentation → Lib-INVENT decoration → scoring**

This notebook takes molecules from `prexsyn_sampled.json` (output of `pipeline.ipynb`),
fragments them into scaffolds with attachment points, then uses Lib-INVENT's
`scaffold_decorating` mode to generate decorated analogs.

**Kernel:** `prexsyn` conda environment (Python 3.11, rdkit required).

**Prerequisite:** `data/prexsyn_sampled.json` must exist (run `pipeline.ipynb` Section 4 first).

In [ ]:
import hashlib
import json
import os
import pathlib
import subprocess
import sys

# ── resolve project root ───────────────────────────────────────────────────────
_root = pathlib.Path.cwd()
if not (_root / "src").exists():
    _root = _root.parent

PIPELINE_DIR = _root / "src" / "modifications" / "ml_based" / "pipeline"
EVAL_DIR     = _root / "src" / "evaluation"
sys.path.insert(0, str(PIPELINE_DIR))
sys.path.insert(0, str(_root))

LIB_INVENT_DIR = _root / "src" / "modifications" / "ml_based" / "Lib-INVENT"
INPUT_PY       = LIB_INVENT_DIR / "input.py"

# ── paths ──────────────────────────────────────────────────────────────────────
DATA_DIR             = _root / "data"
SAMPLES_JSON         = DATA_DIR / "prexsyn_sampled.json"
MODEL_PATH           = LIB_INVENT_DIR / "trained_models" / "reaction_based.model"
RUN_DIR              = DATA_DIR / "libinvent_runs" / "prexsyn_decoration"
AIZYNTHFINDER_CONFIG = DATA_DIR / "aizynthfinder" / "config.yml"

# ── settings ───────────────────────────────────────────────────────────────────
INPUT_MODE    = "source"
FRAG_METHOD   = "brics"
N_DECORATIONS = 32

# ── input size controls ────────────────────────────────────────────────────────
N_SOURCE_LIMIT       = None
N_SAMPLES_PER_SOURCE = None
N_MOLECULES_LIMIT    = None

# ── AiZynthFinder ──────────────────────────────────────────────────────────────
N_WORKERS         = max(1, os.cpu_count() - 1)
USE_AIZYNTHFINDER = AIZYNTHFINDER_CONFIG.exists()

RUN_DIR.mkdir(parents=True, exist_ok=True)

# ── Cache invalidation ─────────────────────────────────────────────────────────
# Hash the parameters that affect decorated.csv content.
# If they change, delete the CSV immediately so downstream cells see fresh data.
_run_params = {
    "INPUT_MODE":          INPUT_MODE,
    "FRAG_METHOD":         FRAG_METHOD,
    "N_DECORATIONS":       N_DECORATIONS,
    "N_SOURCE_LIMIT":      N_SOURCE_LIMIT,
    "N_SAMPLES_PER_SOURCE": N_SAMPLES_PER_SOURCE,
    "N_MOLECULES_LIMIT":   N_MOLECULES_LIMIT,
}
_params_hash  = hashlib.md5(json.dumps(_run_params, sort_keys=True).encode()).hexdigest()
_meta_file    = RUN_DIR / ".run_params.json"
_output_csv   = RUN_DIR / "decorated.csv"

_cached_hash  = json.loads(_meta_file.read_text()).get("hash") if _meta_file.exists() else None

if _cached_hash != _params_hash:
    if _output_csv.exists():
        _output_csv.unlink()
        print(f"[cache] Parameters changed — deleted stale decorated.csv")
    _meta_file.write_text(json.dumps({"hash": _params_hash, "params": _run_params}, indent=2))

print(f"Project root        : {_root}")
print(f"Input mode          : {INPUT_MODE}")
print(f"N_SOURCE_LIMIT      : {N_SOURCE_LIMIT}")
print(f"N_SAMPLES_PER_SOURCE: {N_SAMPLES_PER_SOURCE}")
print(f"N_MOLECULES_LIMIT   : {N_MOLECULES_LIMIT}")
print(f"N_DECORATIONS       : {N_DECORATIONS}")
print(f"AiZynthFinder       : {'ENABLED' if USE_AIZYNTHFINDER else 'DISABLED'}")

## 1. Load PrexSyn Samples

Load the JSON produced by `pipeline.ipynb` Section 4.  
Each entry contains a `source_smiles` and a list of `generated_smiles`.

In [ ]:
assert SAMPLES_JSON.exists(), (
    f"PrexSyn samples not found: {SAMPLES_JSON}\n"
    "Run pipeline.ipynb Section 4 first."
)

with open(SAMPLES_JSON) as f:
    prexsyn_results = json.load(f)

print(f"Loaded {len(prexsyn_results)} source molecules")
print(f"Total generated SMILES: {sum(len(r.get('generated_smiles', [])) for r in prexsyn_results)}")
print(f"Example entry keys: {list(prexsyn_results[0].keys())}")

## 2. Extract Input SMILES

Select molecules to decorate based on `INPUT_MODE`.  
- `"source"` uses the original ChEMBL SMILES — matches PrexSyn's input, enabling direct comparison.  
- `"generated"` uses PrexSyn-generated analogs for sequential modification.

In [ ]:
from rdkit import Chem

def _is_valid(smi: str) -> bool:
    return bool(smi) and Chem.MolFromSmiles(smi) is not None

# Apply N_SOURCE_LIMIT to the JSON entries
entries = prexsyn_results[:N_SOURCE_LIMIT] if N_SOURCE_LIMIT is not None else prexsyn_results

if INPUT_MODE == "source":
    raw_smiles = [r["source_smiles"] for r in entries]
else:  # "generated"
    raw_smiles = [
        smi
        for r in entries
        for smi in (r.get("generated_smiles", [])[:N_SAMPLES_PER_SOURCE]
                    if N_SAMPLES_PER_SOURCE is not None
                    else r.get("generated_smiles", []))
    ]

# Deduplicate and validate
seen = set()
input_smiles = []
for smi in raw_smiles:
    if smi in seen or not _is_valid(smi):
        continue
    seen.add(smi)
    input_smiles.append(smi)

# Apply hard cap
if N_MOLECULES_LIMIT is not None:
    input_smiles = input_smiles[:N_MOLECULES_LIMIT]

print(f"Mode                : {INPUT_MODE}")
print(f"Source entries used : {len(entries)} / {len(prexsyn_results)}")
if INPUT_MODE == "generated":
    print(f"Samples per source  : {N_SAMPLES_PER_SOURCE if N_SAMPLES_PER_SOURCE is not None else 'all'}")
print(f"Raw SMILES          : {len(raw_smiles)}")
print(f"After dedup/valid   : {len(input_smiles)}"
      + (f"  (capped from {len(seen)})" if N_MOLECULES_LIMIT is not None else ""))
print(f"\nExamples:")
for s in input_smiles[:3]:
    print(f"  {s}")

## 3. Fragment into Scaffolds

BRICS or RECAP decomposition breaks each molecule at synthetic bond sites.  
Fragments with 1–3 attachment points (`[*:0]`, `[*:1]`, ...) and ≥ 5 heavy atoms are kept as scaffolds.

In [ ]:
from fragment import get_scaffolds
from tqdm.notebook import tqdm

all_scaffolds = set()
scaffold_map  = {}     # source_smiles -> list of scaffolds
n_failed      = 0

for smi in tqdm(input_smiles, desc="Fragmenting", unit="mol"):
    scaffolds = get_scaffolds(smi, method=FRAG_METHOD)
    scaffold_map[smi] = scaffolds
    if scaffolds:
        all_scaffolds.update(scaffolds)
    else:
        n_failed += 1

all_scaffolds = sorted(all_scaffolds)

print(f"Molecules fragmented : {len(input_smiles)}")
print(f"No scaffolds found   : {n_failed}")
print(f"Total unique scaffolds: {len(all_scaffolds)}")

In [ ]:
# Preview scaffolds with attachment point info
import re

_DUMMY_PAT = re.compile(r"\[\d+\*\]|\[\*\]|\[\*:\d+\]")

print(f"{'Scaffold SMILES':<60} {'#Attach'}")
print("-" * 68)
for sc in all_scaffolds[:15]:
    n_attach = len(_DUMMY_PAT.findall(sc))
    print(f"{sc:<60} {n_attach}")
if len(all_scaffolds) > 15:
    print(f"... and {len(all_scaffolds) - 15} more")

## 4. Run Lib-INVENT Scaffold Decoration

Write scaffolds to a `.smi` file, build the `scaffold_decorating` JSON config,
then invoke `input.py` as a subprocess.

In [ ]:
from configs import make_decorate_config, write_json, write_scaffolds_smi

assert MODEL_PATH.exists(), (
    f"Pretrained model not found: {MODEL_PATH}\n"
    "Expected: data/reaction_based.model"
)
assert INPUT_PY.exists(), (
    f"Lib-INVENT input.py not found: {INPUT_PY}\n"
    "Check that src/modifications/ml_based/Lib-INVENT/ is present."
)

SCAFFOLDS_SMI = RUN_DIR / "scaffolds.smi"
OUTPUT_CSV    = RUN_DIR / "decorated.csv"
LOG_DIR       = RUN_DIR / "logs" / "decorate"
CONFIG_PATH   = RUN_DIR / "decorate_config.json"

write_scaffolds_smi(all_scaffolds, str(SCAFFOLDS_SMI))
print(f"Wrote {len(all_scaffolds)} scaffolds → {SCAFFOLDS_SMI}")

config = make_decorate_config(
    model_path         = str(MODEL_PATH.resolve()),
    scaffolds_smi_path = str(SCAFFOLDS_SMI.resolve()),
    output_csv_path    = str(OUTPUT_CSV.resolve()),
    logging_path       = str(LOG_DIR.resolve()),
    batch_size         = 64,
    n_decorations      = N_DECORATIONS,
)
write_json(config, str(CONFIG_PATH))
print(f"Config written → {CONFIG_PATH}")
print(json.dumps(config, indent=2))

In [ ]:
import io

# Run Lib-INVENT (scaffold_decorating)
cmd = [sys.executable, str(INPUT_PY), str(CONFIG_PATH)]
print(f"Running: {' '.join(cmd)}\n")

result = subprocess.run(
    cmd,
    cwd    = str(LIB_INVENT_DIR),
    stdout = subprocess.PIPE,
    stderr = subprocess.STDOUT,   # merge stderr into stdout
    text   = True,
)

print(result.stdout or "")
if result.returncode != 0:
    print(f"\n[ERROR] Lib-INVENT exited with code {result.returncode}")
else:
    print(f"\nLib-INVENT finished. Output: {OUTPUT_CSV}")

## 5. Parse and Inspect Decorated Molecules

In [ ]:
import pandas as pd

assert OUTPUT_CSV.exists(), f"Decoration output not found: {OUTPUT_CSV}"

df_dec = pd.read_csv(OUTPUT_CSV)   # comma-separated (Lib-INVENT default)
print(f"Raw rows      : {len(df_dec)}")
print(f"Columns       : {list(df_dec.columns)}")
df_dec.head()

In [ ]:
# Detect SMILES column and validate molecules
smiles_col = next(
    (c for c in df_dec.columns if c.lower() in ("smiles", "output_smiles", "molecules")),
    df_dec.columns[0]
)
print(f"SMILES column : '{smiles_col}'")

df_dec["mol_valid"] = df_dec[smiles_col].apply(
    lambda s: Chem.MolFromSmiles(str(s)) is not None if pd.notna(s) else False
)
df_valid = df_dec[df_dec["mol_valid"]].copy()
print(f"Valid molecules: {len(df_valid)} / {len(df_dec)} ({100*len(df_valid)/len(df_dec):.1f}%)")
print(f"Unique SMILES  : {df_valid[smiles_col].nunique()}")

## 6. Score Decorated Molecules

Compute Tanimoto similarity to source molecules and QED drug-likeness for each decoration.

In [ ]:
import numpy as np
from rdkit import DataStructs
from rdkit.Chem import AllChem, QED

def ecfp4(smi: str):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)

def tanimoto(fp1, fp2) -> float:
    if fp1 is None or fp2 is None:
        return 0.0
    return DataStructs.TanimotoSimilarity(fp1, fp2)

def qed_score(smi: str) -> float:
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return float("nan")
    return QED.qed(mol)

# Precompute source fingerprints
source_fps = {smi: ecfp4(smi) for smi in input_smiles}

# For scaffold->source mapping, find which source SMILES produced each scaffold
scaffold_to_sources: dict = {}
for src, scaffolds in scaffold_map.items():
    for sc in scaffolds:
        scaffold_to_sources.setdefault(sc, []).append(src)

print(f"Source fingerprints computed: {len(source_fps)}")

In [ ]:
from tqdm.notebook import tqdm

# Column name mapping (Lib-INVENT output columns)
NLL_COL  = "Likelihoods"
DEC_COL  = "Decorations"

rows_scored = []

for _, row in tqdm(df_valid.iterrows(), total=len(df_valid), desc="Scoring", unit="mol"):
    dec_smi  = str(row[smiles_col])
    scaffold = str(row.get("Scaffold", ""))
    nll      = float(row[NLL_COL]) if NLL_COL in row else float("nan")

    dec_fp = ecfp4(dec_smi)
    qed    = qed_score(dec_smi)

    # Tanimoto against the source(s) that produced this scaffold
    sources = scaffold_to_sources.get(scaffold, list(source_fps.keys())[:1])
    best_tanimoto = max(
        (tanimoto(dec_fp, source_fps[src]) for src in sources if src in source_fps),
        default=0.0,
    )

    rows_scored.append({
        "scaffold"    : scaffold,
        "decoration"  : row.get(DEC_COL, ""),
        "smiles"      : dec_smi,
        "nll"         : nll,
        "tanimoto"    : best_tanimoto,
        "qed"         : qed,
        "is_similar"  : best_tanimoto >= 0.3,
        "is_drug_like": qed >= 0.5,
    })

df_scored = pd.DataFrame(rows_scored)
df_scored["is_hit"] = df_scored["is_similar"] & df_scored["is_drug_like"]

output_path = RUN_DIR / "libinvent_scores.csv"
df_scored.to_csv(output_path, index=False)
print(f"Saved {len(df_scored)} rows → {output_path}")

## 8. Results Summary

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.notebook import tqdm as tqdm_nb

if USE_AIZYNTHFINDER:
    from src.evaluation.synth_parallel import worker_init, score_one

    candidates = df_scored["smiles"].dropna().unique().tolist()
    synth_results: dict[str, bool] = {}

    with ProcessPoolExecutor(
        max_workers=N_WORKERS,
        initializer=worker_init,
        initargs=(str(AIZYNTHFINDER_CONFIG),),
    ) as executor:
        futures = {executor.submit(score_one, smi): smi for smi in candidates}
        for future in tqdm_nb(as_completed(futures), total=len(futures),
                              desc="AiZynthFinder", unit="mol"):
            smi, is_solved = future.result()
            synth_results[smi] = is_solved

    df_scored["is_synth"] = df_scored["smiles"].map(synth_results).fillna(False)

else:
    # Config not found — skip gate, treat all molecules as synthesizable
    print("[WARN] AiZynthFinder config not found. Setting is_synth=True for all molecules.")
    df_scored["is_synth"] = True

n_synth = df_scored["is_synth"].sum()
print(f"Synthesizable (is_solved): {n_synth} / {len(df_scored)} ({100 * n_synth / len(df_scored):.1f}%)")

# Update hit definition: similar + drug-like + synthesizable
df_scored["is_hit"] = df_scored["is_similar"] & df_scored["is_drug_like"] & df_scored["is_synth"]

# Persist updated scores
output_path = RUN_DIR / "libinvent_scores.csv"
df_scored.to_csv(output_path, index=False)
print(f"Scores saved → {output_path}")

In [ ]:
total = len(df_scored)
hits  = df_scored["is_hit"].sum()

print(f"Total decorated molecules : {total}")
print(f"Valid (RDKit parseable)   : {len(df_valid)}")
print(f"Unique SMILES             : {df_scored['smiles'].nunique()}")
print()
print(f"Mean Tanimoto to source   : {df_scored['tanimoto'].mean():.3f}")
print(f"Mean QED                  : {df_scored['qed'].mean():.3f}")
print()
print(f"Similar    (Tanimoto ≥ 0.3)          : {df_scored['is_similar'].sum():>6}  ({100*df_scored['is_similar'].mean():.1f}%)")
print(f"Drug-like  (QED ≥ 0.5)               : {df_scored['is_drug_like'].sum():>6}  ({100*df_scored['is_drug_like'].mean():.1f}%)")
print(f"Synthesizable (AiZynthFinder solved) : {df_scored['is_synth'].sum():>6}  ({100*df_scored['is_synth'].mean():.1f}%)")
print()
print(f"Hits (similar + drug-like + synth)   : {hits:>6} / {total}  ({100*hits/total:.1f}%)")

In [ ]:
df_scored[["tanimoto", "qed", "nll", "is_hit"]].describe().round(3)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

df_scored["tanimoto"].hist(bins=40, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].axvline(0.3, color="red", linestyle="--", label="threshold (0.3)")
axes[0].set_title("Tanimoto Similarity")
axes[0].set_xlabel("Tanimoto")
axes[0].legend()

df_scored["qed"].hist(bins=40, ax=axes[1], color="seagreen", edgecolor="white")
axes[1].axvline(0.5, color="red", linestyle="--", label="threshold (0.5)")
axes[1].set_title("QED (Drug-likeness)")
axes[1].set_xlabel("QED")
axes[1].legend()

df_scored["nll"].dropna().hist(bins=40, ax=axes[2], color="darkorange", edgecolor="white")
axes[2].set_title("Negative Log-Likelihood")
axes[2].set_xlabel("NLL")

synth_counts = df_scored["is_synth"].value_counts()
axes[3].bar(
    ["Synthesizable", "Not synthesizable"],
    [synth_counts.get(True, 0), synth_counts.get(False, 0)],
    color=["seagreen", "tomato"], edgecolor="white",
)
axes[3].set_title("AiZynthFinder Gate")
axes[3].set_ylabel("Count")

plt.tight_layout()
plt.savefig(RUN_DIR / "libinvent_score_distributions.png", dpi=150)
plt.show()

In [ ]:
# Top-20 hits by QED
df_scored[df_scored["is_hit"]].sort_values("qed", ascending=False).head(20)[
    ["smiles", "scaffold", "tanimoto", "qed", "nll"]
].reset_index(drop=True)